In [0]:
df_sales_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/workspace/default/raw_data/sales_raw.csv")
)

display(df_sales_bronze)

In [0]:
df_sales_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.pharma_bronze.bronze_sales")

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_sales_silver = (
    df_sales_bronze
    .withColumn(
        "revenue",
        col("quantity_sold") * col("unit_price")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

display(df_sales_silver)

In [0]:
from delta.tables import DeltaTable

delta_sales_target = DeltaTable.forName(
    spark,
    "workspace.pharma_silver.silver_sales"
)

(
    delta_sales_target.alias("target")
    .merge(
        df_sales_silver.alias("source"),
        "target.sale_id = source.sale_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
df_sales_final = spark.table(
    "workspace.pharma_silver.silver_sales"
)

print("Sales Silver records:", df_sales_final.count())

print(
    "Duplicate sale IDs:",
    df_sales_final.groupBy("sale_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
null_sale_ids = df_sales_silver.filter(
    col("sale_id").isNull()
).count()

print("NULL Sale IDs:", null_sale_ids)

In [0]:
duplicate_sale_ids = (
    df_sales_silver
    .groupBy("sale_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Sale IDs:", duplicate_sale_ids)

In [0]:
negative_sales_quantity = df_sales_silver.filter(
    col("quantity_sold") < 0
).count()

print("Negative Sales Quantity:", negative_sales_quantity)